# LunarLander training with a manual cyclic-waypoint automaton

This self-contained Kaggle notebook installs the dependencies, reconstructs the
configurable cyclic-waypoint framework, trains the DDQN agent, generates the
automaton and potential plots, inspects the metrics, and packages every output.

The task repeatedly enforces the ordered sequence declared in `waypoint_cycle`.
Reaching its last waypoint produces one synthetic reward and triggers an
immediate epsilon reset to `q1` in the same Gymnasium step. The transient
accepting state is never exposed to the network and does not have a potential
heatmap. Acceptance does not end the Gymnasium episode.

Enable a GPU from **Settings → Accelerator → GPU** before starting.


## 1. Install system and Python dependencies


In [ ]:
!apt-get update -qq
!apt-get install -y -qq graphviz swig
%pip install -q "gymnasium[box2d]" graphviz pandas matplotlib


## 2. Create the writable project directory


In [ ]:
from pathlib import Path
import os

WORK_DIR = Path("/kaggle/working/manual_experiment")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {WORK_DIR}")


## 3. Write the manual automaton module


In [ ]:
%%writefile manual_automaton.py
"""Manual finite-state automata used by the experiment."""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class AutomatonStep:
    """Result of one valuation, after exhausting automatic transitions."""

    next_state: str
    completed_cycle: bool = False
    reached_waypoint: str | None = None


class CyclicWaypointsAutomaton:
    """Automaton that repeatedly visits an ordered sequence of waypoints.

    Every stable state ``qi`` waits for the waypoint at position ``i`` in
    ``waypoint_cycle``. Reaching the last waypoint enters a transient accepting
    state and immediately resets to ``q1`` without consuming another
    environment step.
    """

    def __init__(self, waypoint_cycle: Sequence[str]):
        if isinstance(waypoint_cycle, (str, bytes)):
            raise TypeError("waypoint_cycle must be a sequence of proposition names")

        cycle = tuple(waypoint_cycle)
        if not cycle:
            raise ValueError("waypoint_cycle must contain at least one waypoint")
        if any(not isinstance(name, str) or not name for name in cycle):
            raise ValueError("Waypoint proposition names must be non-empty strings")
        if len(set(cycle)) != len(cycle):
            raise ValueError("waypoint_cycle cannot contain duplicate propositions")

        self.waypoint_cycle = cycle
        self.active_states = tuple(f"q{index}" for index in range(1, len(cycle) + 1))
        self.accepting_state = f"q{len(cycle) + 1}"
        self.states = (*self.active_states, self.accepting_state)
        self.initial_state = self.active_states[0]
        self.accepting_states = frozenset({self.accepting_state})
        self.num_phases = len(self.active_states)
        self._state_to_index = {
            state: index for index, state in enumerate(self.active_states)
        }

    @property
    def required_propositions(self) -> frozenset[str]:
        """Return the waypoint propositions consumed by this automaton."""
        return frozenset(self.waypoint_cycle)

    def get_initial_q(self) -> str:
        """Return the state active at the beginning of every episode."""
        return self.initial_state

    def is_accepting(self, state: str) -> bool:
        """Return whether ``state`` is the transient accepting state."""
        self._validate_state(state)
        return state in self.accepting_states

    def expected_waypoint(self, state: str) -> str:
        """Return the waypoint proposition awaited in a stable state."""
        self._validate_state(state)
        if state == self.accepting_state:
            raise ValueError("The transient accepting state does not await a waypoint")
        return self.waypoint_cycle[self._state_to_index[state]]

    def advance(self, current_q: str, truth_assignment: Mapping[str, bool]) -> AutomatonStep:
        """Consume one valuation and exhaust the accepting epsilon transition."""
        self._validate_state(current_q)

        # Also close the accepting state when explicitly supplied by diagnostics
        # or external code.
        if current_q == self.accepting_state:
            return AutomatonStep(self.initial_state)

        phase_index = self._state_to_index[current_q]
        expected = self.waypoint_cycle[phase_index]
        if not bool(truth_assignment.get(expected, False)):
            return AutomatonStep(current_q)

        if phase_index == len(self.waypoint_cycle) - 1:
            return AutomatonStep(
                self.initial_state,
                completed_cycle=True,
                reached_waypoint=expected,
            )

        return AutomatonStep(
            self.active_states[phase_index + 1],
            reached_waypoint=expected,
        )

    def get_next_q(self, current_q: str, truth_assignment: Mapping[str, bool]) -> str:
        """Return the stable state reached after applying epsilon closure."""
        return self.advance(current_q, truth_assignment).next_state

    def validate_waypoints(self, waypoints: Mapping[str, tuple[int, int]], width: int, height: int) -> None:
        """Validate the propositions and coordinates required by the automaton."""
        if width <= 0 or height <= 0:
            raise ValueError("Grid dimensions must be positive")

        missing = sorted(self.required_propositions - set(waypoints))
        if missing:
            raise ValueError(f"Missing required cycle waypoints: {missing}")

        for name, coordinates in waypoints.items():
            if not isinstance(coordinates, (tuple, list)) or len(coordinates) != 2:
                raise ValueError(
                    f"Waypoint {name!r} must contain exactly two coordinates"
                )
            x, y = coordinates
            if not isinstance(x, int) or not isinstance(y, int):
                raise ValueError(f"Waypoint {name!r} coordinates must be integers")
            if not 0 <= x < width or not 0 <= y < height:
                raise ValueError(
                    f"Waypoint {name!r} at ({x}, {y}) is outside "
                    f"the {width}x{height} grid"
                )

    def describe_cycle(self) -> str:
        """Return a compact human-readable representation of the automaton."""
        transitions = [
            f"{state} --{waypoint}{'/reward' if index + 1 == self.num_phases else ''}--> "
            f"{self.active_states[index + 1] if index + 1 < self.num_phases else self.accepting_state}"
            for index, (state, waypoint) in enumerate(
                zip(self.active_states, self.waypoint_cycle)
            )
        ]
        transitions.append(f"{self.accepting_state} --epsilon--> {self.initial_state}")
        return " | ".join(transitions)

    def render_graph(self, filename: str = "cyclic_waypoints_automaton", directory: str | Path = "img") -> None:
        """Render a compact diagram when the optional Graphviz package is present."""
        try:
            from graphviz import Source

            lines = [
                "digraph cyclic_waypoints {",
                "    rankdir=LR;",
                "    node [shape=circle];",
                "    start [shape=point];",
                f"    {self.accepting_state} [shape=doublecircle];",
                f"    start -> {self.initial_state};",
            ]
            for index, (state, waypoint) in enumerate(
                zip(self.active_states, self.waypoint_cycle)
            ):
                destination = (
                    self.active_states[index + 1]
                    if index + 1 < self.num_phases
                    else self.accepting_state
                )
                escaped_waypoint = waypoint.replace("\\", "\\\\").replace('"', '\\"')
                reward_suffix = " / reward" if destination == self.accepting_state else ""
                lines.extend(
                    [
                        f'    {state} -> {destination} [label="{escaped_waypoint}{reward_suffix}"];',
                        f'    {state} -> {state} [label="not {escaped_waypoint}"];',
                    ]
                )
            lines.extend(
                [
                    f'    {self.accepting_state} -> {self.initial_state} [label="epsilon"];',
                    "}",
                ]
            )
            Source("\n".join(lines)).render(
                filename=filename,
                directory=str(directory),
                format="png",
                cleanup=True,
            )
            print(f"Automaton graph saved to: {directory}/{filename}.png")
        except Exception as error:
            print(f"[Graphviz error] Could not render the automaton graph: {error}")

    def _validate_state(self, state: str) -> None:
        if state not in self.states:
            raise ValueError(f"Unknown automaton state {state!r}")


class AlternatingGoalsAutomaton(CyclicWaypointsAutomaton):
    """Backward-compatible two-waypoint specialization."""

    WAITING_FOR_G1 = "q1"
    WAITING_FOR_G2 = "q2"
    ACCEPTING = "q3"

    def __init__(self, first_goal: str = "g1", second_goal: str = "g2"):
        super().__init__((first_goal, second_goal))
        self.first_goal = first_goal
        self.second_goal = second_goal

    def render_graph(self, filename: str = "alternating_goals_automaton", directory: str | Path = "img") -> None:
        """Render using the historical default filename."""
        super().render_graph(filename=filename, directory=directory)


## 4. Write the continuing abstract MDP


In [ ]:
%%writefile abstract_mdps.py
"""Abstract grid MDP driven by a manually defined automaton."""

from collections import defaultdict


class ManualWaypointMDP:
    """Grid abstraction whose state is ``(x, y, q)``."""

    def __init__(
        self,
        waypoints_dict,
        automaton,
        width=12,
        height=12,
        gamma=0.99,
        goal_reward=10000,
    ):
        self.width = width
        self.height = height
        self.gamma = gamma
        self.actions = [0, 1, 2, 3, 4, 5, 6, 7]
        self.waypoints_dict = waypoints_dict
        self.automaton = automaton
        self.num_phases = self.automaton.num_phases
        self.states = [
            (x, y, q)
            for x in range(width)
            for y in range(height)
            for q in self.automaton.active_states
        ]
        self.goal_reward = goal_reward
        self.v_star = defaultdict(float)

    def _get_truth_assignment(self, x, y):
        """Map grid coordinates to waypoint proposition values."""
        return {
            proposition: (x == waypoint_x and y == waypoint_y)
            for proposition, (waypoint_x, waypoint_y) in self.waypoints_dict.items()
        }

    def get_transitions(self, state, action):
        """Apply one abstract movement and one automaton transition."""
        x, y, q = state

        next_y = y
        if action in [0, 4, 5]:
            next_y = min(y + 1, self.height - 1)
        elif action in [1, 6, 7]:
            next_y = max(y - 1, 0)

        next_x = x
        if action in [2, 4, 6]:
            next_x = max(x - 1, 0)
        elif action in [3, 5, 7]:
            next_x = min(x + 1, self.width - 1)

        truth_assignment = self._get_truth_assignment(next_x, next_y)
        automaton_step = self.automaton.advance(q, truth_assignment)
        reward = float(self.goal_reward) if automaton_step.completed_cycle else 0.0
        return (next_x, next_y, automaton_step.next_state), reward

    def print_policy(self):
        arrows = {
            0: "↑",
            1: "↓",
            2: "←",
            3: "→",
            4: "↖",
            5: "↗",
            6: "↙",
            7: "↘",
        }

        for q in self.automaton.active_states:
            print(f"\n===== POLICY - AUTOMATON STATE q={q} =====")
            for y in reversed(range(self.height)):
                row = []
                for x in range(self.width):
                    state = (x, y, q)
                    best_action = max(
                        self.actions,
                        key=lambda action: (
                            self.get_transitions(state, action)[1]
                            + self.gamma
                            * self.v_star[self.get_transitions(state, action)[0]]
                        ),
                    )
                    row.append(f" {arrows[best_action]} ")
                print("".join(row))

    def value_iteration(self, theta=0.001):
        """Compute continuing-task values, including every future goal cycle."""
        print("Value Iteration...")

        while True:
            delta = 0.0
            new_v = self.v_star.copy()
            for state in self.states:
                action_values = []
                for action in self.actions:
                    next_state, reward = self.get_transitions(state, action)
                    action_values.append(
                        reward + self.gamma * self.v_star[next_state]
                    )
                best_value = max(action_values)
                delta = max(delta, abs(best_value - self.v_star[state]))
                new_v[state] = best_value
            self.v_star = new_v
            if delta < theta:
                break

        self.print_policy()


## 5. Write the DDQN agent module


In [ ]:
%%writefile agent.py
import numpy as np
import random as ran
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

class DuelingQNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DuelingQNetwork, self).__init__()
        self.feature = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
        )
        self.value_stream = nn.Linear(128, 1)
        self.advantage_stream = nn.Linear(128, action_dim)

    def forward(self, x):
        features = self.feature(x)
        value = self.value_stream(features)
        advantages = self.advantage_stream(features)
        return value + advantages - advantages.mean(dim=1, keepdim=True)

class ReplayBuffer:
    def __init__(self, capacity, num_phases):
        if num_phases < 0:
            raise ValueError("num_phases cannot be negative")
        self.capacity = capacity
        self.num_phases = num_phases
        self.buffer = []
        self.phase_indices = []
        self.phase_counts = np.zeros(num_phases, dtype=np.int64)
        self.position = 0

    def push(self, state, action, reward, next_state, done):
        """Insert a transition and update automaton-state counts."""
        transition = (state, action, reward, next_state, done)
        phase_index = (
            int(np.argmax(state[-self.num_phases:]))
            if self.num_phases > 0
            else None
        )

        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
            self.phase_indices.append(phase_index)
        else:
            replaced_phase_index = self.phase_indices[self.position]
            if replaced_phase_index is not None:
                self.phase_counts[replaced_phase_index] -= 1
            self.buffer[self.position] = transition
            self.phase_indices[self.position] = phase_index

        if phase_index is not None:
            self.phase_counts[phase_index] += 1
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        """Sample transitions efficiently from the indexable ring buffer."""
        batch = ran.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.array, zip(*batch))
        return state, action, reward, next_state, done

    def __len__(self):
        return len(self.buffer)

    def q_fraction_onehot(self, q_index, num_phases):
        """Return an automaton-state fraction from maintained counts."""
        if num_phases != self.num_phases:
            raise ValueError(
                f"Expected {self.num_phases} automaton states, received {num_phases}"
            )
        if not 0 <= q_index < self.num_phases:
            raise IndexError(f"Automaton state index {q_index} is out of range")
        if len(self.buffer) == 0:
            return 0.0
        return float(self.phase_counts[q_index] / len(self.buffer))

class HierarchicalDQNLearner:
    def __init__(
        self,
        env,
        abstract_mdp=None,
        max_episodes=1000,
        eps_decay=0.995,
        gamma=0.99,
        policy_name="policy",
        extra_state_dims=0,
        use_polyak=True,
        tau=0.005,
        target_update_freq=1000,
        network_type="standard",
    ):
        if extra_state_dims < 0:
            raise ValueError("extra_state_dims cannot be negative")
        if not 0.0 < tau <= 1.0:
            raise ValueError("tau must be in the interval (0, 1]")
        if target_update_freq <= 0:
            raise ValueError("target_update_freq must be greater than zero")
        if network_type not in {"standard", "dueling"}:
            raise ValueError("network_type must be one of: standard, dueling")

        self.env = env
        self.abstract_mdp = abstract_mdp
        self.max_episodes = max_episodes
        self.gamma = gamma
        self.policy_name = policy_name
        self.network_type = network_type
        self.algo_name = "Dueling DDQN" if network_type == "dueling" else "DDQN"
        
        self.batch_size = 64
        self.lr = 1e-3
        self.use_polyak = use_polyak
        self.tau = tau
        self.target_update_freq = target_update_freq
        self.optimization_steps = 0
        self.eps = 1.0
        self.eps_min = 0.01
        self.eps_decay = eps_decay
        
        # Account for dynamic one-hot phases appended to state
        state_dim = self.env.observation_space.shape[0] + extra_state_dims
        action_dim = self.env.action_space.n
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        network_cls = DuelingQNetwork if network_type == "dueling" else QNetwork
        self.policy_net = network_cls(state_dim, action_dim).to(self.device)
        print(f"Using device:{self.device}")
        
        self.target_net = network_cls(state_dim, action_dim).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.lr)
        self.memory = ReplayBuffer(capacity=300000, num_phases=extra_state_dims)

    def select_action(self, state):
        if ran.random() < self.eps:
            return self.env.action_space.sample()
        else:
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                q_values = self.policy_net(state_tensor)
                return q_values.argmax(dim=1).item()

    def optimize_model(self):
        if len(self.memory) < self.batch_size: return
            
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(self.device)
        
        q_values = self.policy_net(states).gather(1, actions)
        
        with torch.no_grad():
            best_actions = self.policy_net(next_states).argmax(dim=1).unsqueeze(1)
            next_q_values = self.target_net(next_states).gather(1, best_actions)
            target_q_values = rewards + (1 - dones) * self.gamma * next_q_values
            
        loss = F.mse_loss(q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        self.optimization_steps += 1
        if self.use_polyak:
            for target_param, policy_param in zip(self.target_net.parameters(), self.policy_net.parameters()):
                target_param.data.copy_(self.tau * policy_param.data + (1.0 - self.tau) * target_param.data)
        elif self.optimization_steps % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

    def _save_policy(self):
        os.makedirs("./policy", exist_ok=True)
        torch.save(self.policy_net.state_dict(), f"./policy/{self.policy_name}")


## 6. Write plotting and abstraction utilities


In [ ]:
%%writefile utils.py
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle


# Legacy discretization. To reactivate it, uncomment this function and comment
# out the active phi_mapping_grid implementation immediately below.
#
# def phi_mapping_grid(obs, grid_w=12, grid_h=12):
#     """Map coordinates using the original grid_size - 1 discretization."""
#     x, y = float(obs[0]), float(obs[1])
#     abstract_x = int(np.clip((x + 1.0) / 2.0 * (grid_w - 1), 0, grid_w - 1))
#     abstract_y = int(np.clip(y / 1.5 * (grid_h - 1), 0, grid_h - 1))
#     return abstract_x, abstract_y


def phi_mapping_grid(obs, grid_w=12, grid_h=12):
    """Map LunarLander coordinates to uniform bins over x=[-1,1], y=[0,1.5]."""
    if grid_w <= 0 or grid_h <= 0:
        raise ValueError("grid_w and grid_h must be positive")

    x, y = float(obs[0]), float(obs[1])
    abstract_x = int(np.floor((x + 1.0) / 2.0 * grid_w))
    abstract_y = int(np.floor(y / 1.5 * grid_h))
    abstract_x = int(np.clip(abstract_x, 0, grid_w - 1))
    abstract_y = int(np.clip(abstract_y, 0, grid_h - 1))
    return abstract_x, abstract_y


def _axis_boundaries(map_axis, size, lower, upper):
    """Infer bin boundaries from the active mapper."""
    if size <= 0:
        raise ValueError("grid dimensions must be positive")

    boundaries = [float(lower)]
    iterations = 60
    for target_index in range(1, size):
        left, right = float(lower), float(upper)
        for _ in range(iterations):
            midpoint = (left + right) / 2.0
            if map_axis(midpoint) < target_index:
                left = midpoint
            else:
                right = midpoint
        boundaries.append(right)
    boundaries.append(float(upper))
    return np.asarray(boundaries, dtype=float)


def spatial_grid_boundaries(grid_w=12, grid_h=12):
    """Return x/y bin boundaries implied by the active phi_mapping_grid."""
    x_boundaries = _axis_boundaries(
        lambda x: phi_mapping_grid((x, 0.0), grid_w, grid_h)[0],
        grid_w,
        -1.0,
        1.0,
    )
    y_boundaries = _axis_boundaries(
        lambda y: phi_mapping_grid((0.0, y), grid_w, grid_h)[1],
        grid_h,
        0.0,
        1.5,
    )
    return x_boundaries, y_boundaries


def phi_mapping_sequential(obs, q, grid_w=12, grid_h=12):
    abstract_x, abstract_y = phi_mapping_grid(obs, grid_w, grid_h)
    return abstract_x, abstract_y, q


def lunar_lander_visible_observation_bounds():
    """Return the normalised x/y bounds covered by LunarLander's RGB viewport."""
    from gymnasium.envs.box2d import lunar_lander

    viewport_world_width = lunar_lander.VIEWPORT_W / lunar_lander.SCALE
    viewport_world_height = lunar_lander.VIEWPORT_H / lunar_lander.SCALE
    helipad_y = viewport_world_height / 4.0
    lander_y_offset = helipad_y + lunar_lander.LEG_DOWN / lunar_lander.SCALE
    half_world_height = viewport_world_height / 2.0
    visible_y_min = (0.0 - lander_y_offset) / half_world_height
    visible_y_max = (viewport_world_height - lander_y_offset) / half_world_height
    return -1.0, 1.0, visible_y_min, visible_y_max


def _draw_visible_area_overlay(axis, width, height):
    """Mark which portion of the active abstract grid lies in the RGB viewport."""
    visible_x_min, visible_x_max, visible_y_min, visible_y_max = (
        lunar_lander_visible_observation_bounds()
    )
    x_boundaries, y_boundaries = spatial_grid_boundaries(width, height)

    def to_plot(value, boundaries):
        value = float(np.clip(value, boundaries[0], boundaries[-1]))
        index = int(np.searchsorted(boundaries, value, side="right") - 1)
        index = int(np.clip(index, 0, len(boundaries) - 2))
        lower, upper = boundaries[index], boundaries[index + 1]
        fraction = 0.0 if upper <= lower else (value - lower) / (upper - lower)
        return index - 0.5 + fraction

    left = float(to_plot(visible_x_min, x_boundaries))
    right = float(to_plot(visible_x_max, x_boundaries))
    bottom = float(to_plot(visible_y_min, y_boundaries))
    top = float(to_plot(visible_y_max, y_boundaries))

    axis.add_patch(
        Rectangle(
            (left, bottom),
            right - left,
            top - bottom,
            fill=False,
            edgecolor="#ff1744",
            linewidth=1.4,
            linestyle="--",
            label="Visible RGB viewport",
            zorder=5,
        )
    )
    axis.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)

def save_sequential_heatmaps(abstract_mdp, filename_prefix="v_star"):
    """
    Generates and saves a separate heatmap for V* for each phase defined in the MDP,
    without any waypoint or goal markers (clean heatmap).
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    # Store every heatmap directly under img/heatmaps.
    output_dir = os.path.join("img", "heatmaps")
    os.makedirs(output_dir, exist_ok=True)
    filename_prefix = os.path.basename(filename_prefix)
    
    width, height = abstract_mdp.width, abstract_mdp.height
    
    # Extract global min/max for consistent colormap scaling
    all_values = np.array(list(abstract_mdp.v_star.values()))
    computed_vmin = all_values.min() if len(all_values) > 0 else 0
    computed_vmax = all_values.max() if len(all_values) > 0 else 1

    for current_q in abstract_mdp.automaton.active_states:
        matrix = np.zeros((height, width))
        for (x, y, q), value in abstract_mdp.v_star.items():
            if q == current_q and 0 <= x < width and 0 <= y < height:
                matrix[y, x] = value
                
        plt.figure(figsize=(9, 8))
        im = plt.imshow(matrix, cmap='viridis', origin='lower', vmin=computed_vmin, vmax=computed_vmax)
        
        for y in range(height):
            for x in range(width):
                val = matrix[y, x]
                if val > 0.0: 
                    text_color = 'white' if val < (computed_vmax / 2) else 'black'
                    plt.text(x, y, f"{val:.1f}", ha='center', va='center', color=text_color, fontsize=7)
                    
        plt.colorbar(im, fraction=0.046, pad=0.04, label="Potential Value (V*)")
        
        is_accepting = abstract_mdp.automaton.is_accepting(current_q)
        phase_label = "Cycle completed" if is_accepting else "Seeking target"
        plt.title(f"Potential Map (V*) - Automaton State {current_q} ({phase_label})", fontsize=14, fontweight='bold')
        
        ax = plt.gca()
        ax.set_xticks(np.arange(-.5, width, 1), minor=True)
        ax.set_yticks(np.arange(-.5, height, 1), minor=True)
        ax.grid(which='minor', color='w', linestyle='-', linewidth=1, alpha=0.4)
        _draw_visible_area_overlay(ax, width, height)
        
        # Keep the heatmap free of waypoint and goal markers.
            
        plt.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
        plt.savefig(os.path.join(output_dir, f"{filename_prefix}_q{current_q}.png"), dpi=150, bbox_inches='tight')
        plt.close()
        print(f" -> Generated V* Heatmap for automaton state {current_q}")

def plot_comparison_curves(baseline_rewards, shaping_rewards, epsilon_history=None, window_size=100, filename="img/baseline_vs_shaping.png", title="Learning Curve Comparison", baseline_label="Baseline", shaping_label="Shaping"):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax1 = plt.subplots(figsize=(12, 7))
    baseline_ma = pd.Series(baseline_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
    shaping_ma = pd.Series(shaping_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
    x_axis = np.arange(len(baseline_rewards))
    ax1.plot(x_axis, baseline_ma, color='black', linestyle='-', linewidth=2, label=baseline_label)
    ax1.plot(x_axis, shaping_ma, color='blue', linestyle='-', linewidth=2.5, label=shaping_label)
    ax1.set_title(title, fontsize=15, fontweight='bold')
    ax1.set_xlabel(f"Episode # (Moving Average Window = {window_size})", fontsize=12)
    ax1.set_ylabel("Episode Reward", fontsize=12)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    if epsilon_history:
        ax2 = ax1.twinx()
        ax2.plot(x_axis, epsilon_history, color='orange', linestyle='--', linewidth=1.8, label='Epsilon Decay')
        ax2.set_ylabel("Exploration Rate (ε)", color='orange', fontsize=12)
        ax2.tick_params(axis='y', labelcolor='orange')
        ax2.set_ylim(0, 1.05)
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right", fontsize=11)
    else:
        ax1.legend(loc="lower right", fontsize=11)

    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    print(f"\n>>> Comparison plot successfully saved to: {filename}")
    plt.close(fig)

def plot_mean_std_curves(reward_histories_single=None, reward_histories_multi=None, window_size=100, title="Mean Performance with Variance", filename="img/mean_std_plot.png"):
    """
    Plots the mean and standard deviation of reward histories for one or two sets of runs.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax = plt.subplots(figsize=(12, 7))

    def plot_single_curve(reward_histories, label, color):
        if not reward_histories:
            return
        
        # Ensure all histories have the same length by padding with NaNs if necessary
        max_len = max(len(h) for h in reward_histories)
        padded_histories = [np.pad(h, (0, max_len - len(h)), 'constant', constant_values=np.nan) for h in reward_histories]
        
        rewards_df = pd.DataFrame(padded_histories).T
        mean_rewards = rewards_df.mean(axis=1)
        std_rewards = rewards_df.std(axis=1)

        # Apply moving average
        mean_ma = mean_rewards.rolling(window=window_size, min_periods=1, center=True).mean()
        std_ma = std_rewards.rolling(window=window_size, min_periods=1, center=True).mean()

        x_axis = np.arange(len(mean_ma))
        ax.plot(x_axis, mean_ma, label=f"Mean {label}", color=color, linewidth=2.5)
        ax.fill_between(x_axis, mean_ma - std_ma, mean_ma + std_ma, color=color, alpha=0.2, label=f"Std Dev {label}")

    if reward_histories_single:
        plot_single_curve(reward_histories_single, "Single Epsilon", "black")

    if reward_histories_multi:
        plot_single_curve(reward_histories_multi, "Multi Epsilon", "blue")

    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.set_xlabel(f"Episode # (Moving Average Window = {window_size})", fontsize=12)
    ax.set_ylabel("Mean Episode Reward", fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc="lower right", fontsize=11)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    print(f"\n>>> Mean/Std plot successfully saved to: {filename}")
    plt.close(fig)

def plot_buffer_fractions(buffer_histories, window_size=100, filename="img/buffer_fractions.png", state_labels=None):
    """
    Plots the replay buffer composition for N phases dynamically.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    x_axis = np.arange(len(buffer_histories[0]))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(buffer_histories)))
    for idx, history in enumerate(buffer_histories):
        ma = pd.Series(history).rolling(window=window_size, min_periods=1, center=True).mean()
        state_label = state_labels[idx] if state_labels is not None else idx
        ax.plot(x_axis, ma, color=colors[idx], linewidth=2.5, label=f'Automaton state {state_label}')
    
    ax.set_title(f"Replay Buffer Composition (MA Window = {window_size})", fontsize=14, fontweight='bold')
    ax.set_ylabel("Fraction in Buffer", fontsize=12)
    ax.set_ylim(0, 1.05)
    
    ideal_balance = 1.0 / len(buffer_histories)
    ax.axhline(y=ideal_balance, color='gray', linestyle=':', alpha=0.7, label=f'Ideal Balance ({ideal_balance:.0%})')
    
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=len(buffer_histories)+1, fontsize=11)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    plt.close(fig)

def plot_shaping_reward_breakdown(true_rewards, total_rewards, eps_histories, window_size=100, filename="img/shaping_reward_breakdown.png"):
    """
    Plots the moving average of rewards (True vs Total) and overlays the N-phase Epsilon decay dynamically.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax1 = plt.subplots(figsize=(12, 7))
    
    # Moving Average Calculation
    if len(true_rewards) >= window_size:
        true_ma = pd.Series(true_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
        total_ma = pd.Series(total_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
    else:
        true_ma = true_rewards
        total_ma = total_rewards
        
    x_axis = np.arange(len(true_rewards))
        
    # Plot Rewards (Left Y-Axis)
    ax1.plot(x_axis, true_ma, color='green', linestyle='-', linewidth=2, label='Synthetic Goal Reward')
    ax1.plot(x_axis, total_ma, color='purple', linestyle='-', linewidth=2.5, label='Learning Reward (Goal + Shaping)')
    
    ax1.set_title(f"Shaping Agent Reward Analysis (MA Window = {window_size})", fontsize=15, fontweight='bold')
    ax1.set_xlabel("Episode #", fontsize=12)
    ax1.set_ylabel("Episode Reward", fontsize=12)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # Plot Epsilon Decays (Right Y-Axis)
    ax2 = ax1.twinx()
    
    # Check if eps_histories is a list of lists/arrays (multi-epsilon case)
    is_multi_eps = any(isinstance(i, (list, np.ndarray)) for i in eps_histories)

    if is_multi_eps:
        num_phases = len(eps_histories)
        colors = plt.cm.plasma(np.linspace(0, 0.8, num_phases))
        for idx in range(num_phases):
            label = "Goal" if idx == num_phases - 1 else f"WP {idx + 1}"
            ax2.plot(x_axis, eps_histories[idx], color=colors[idx], linestyle='--', linewidth=2, alpha=0.8, label=f'ε Decay (q={idx}: {label})')
    else: # Single epsilon history
        ax2.plot(x_axis, eps_histories, color='orange', linestyle='--', linewidth=1.8, label='Epsilon Decay')

    # Align the zero of both y-axes for better visual comparison
    y1_min, y1_max = ax1.get_ylim()
    y2_min, y2_max = -0.05, 1.05 # Epsilon range is fixed
    
    # Align y-axes so that the zero points match.
    if y1_min < 0 < y1_max:
        # Calculate the proportional position of zero on the reward axis
        zero_ratio = -y1_min / (y1_max - y1_min)
        # Set the epsilon axis limits so its zero is at the same ratio
        new_y2_min = -zero_ratio * y2_max / (1 - zero_ratio)
        ax2.set_ylim(new_y2_min, y2_max)
    else:
        ax2.set_ylim(y2_min, y2_max)

    ax2.set_ylabel("Exploration Rate (ε)", color='black', fontsize=12)

    # Combine Legends from both axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    
    # Dynamically calculate legend columns based on number of items to keep it compact
    legend_cols = max(2, (len(labels1) + len(labels2)) // 2)
    
    ax1.legend(
        lines1 + lines2, labels1 + labels2, 
        loc="upper center", bbox_to_anchor=(0.5, -0.15), 
        ncol=legend_cols, fontsize=11, framealpha=1.0
    )
    
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    plt.close(fig)


## 7. Write the manual task configuration


In [ ]:
%%writefile trajectory.json
{
  "grid_w": 12,
  "grid_h": 12,
  "goal_reward": 10000,
  "waypoint_cycle": [
    "g1",
    "g2"
  ],
  "waypoints_dict": {
    "g1": [
      1,
      8
    ],
    "g2": [
      9,
      8
    ]
  }
}


## 8. Write the training program

The training loop rewards only an effective transition into the accepting
state. Acceptance is recorded as a completed cycle; only Gymnasium
`terminated` or `truncated` ends an episode.


In [ ]:
%%writefile trainer.py
# ==============================
# Standard library imports
# ==============================

import argparse
import json
import os
from collections import Counter

# ==============================
# External and project imports
# ==============================

import gymnasium as gym
import numpy as np

from abstract_mdps import ManualWaypointMDP
from agent import HierarchicalDQNLearner
from manual_automaton import CyclicWaypointsAutomaton
from utils import phi_mapping_sequential, plot_buffer_fractions, plot_shaping_reward_breakdown, save_sequential_heatmaps


# ==============================
# Data and state helpers
# ==============================

def save_training_data(filename, **kwargs):
    """Convert training metrics to arrays and save them in a compressed NPZ file."""
    # Preserve numeric dtypes and rectangular shapes for direct plotting.
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    np_data = {key: np.asarray(value) for key, value in kwargs.items()}
    if any(array.dtype == object for array in np_data.values()):
        raise ValueError("Training metrics must be rectangular numeric arrays")
    np.savez_compressed(filename, **np_data)
    print(f"\nTraining data saved to: {filename}")


def _abstract_position(observation, abstract_mdp):
    """Map a raw environment observation to its abstract spatial coordinates."""
    x, y, _ = phi_mapping_sequential(
        observation, 0, abstract_mdp.width, abstract_mdp.height
    )
    return x, y


def _augment_state(observation, q, state_to_index):
    """Append a one-hot encoding of the current automaton state."""
    one_hot = np.zeros(len(state_to_index), dtype=np.float32)
    one_hot[state_to_index[q]] = 1.0
    return np.concatenate((observation, one_hot)).astype(np.float32)


def _evaluate_initial_automaton_state(observation, abstract_mdp):
    """Consume the initial observation and return the first active automaton state."""
    initial_x, initial_y = _abstract_position(observation, abstract_mdp)
    initial_truth_assignment = abstract_mdp._get_truth_assignment(initial_x, initial_y)
    pre_trace_q = abstract_mdp.automaton.get_initial_q()
    return abstract_mdp.automaton.advance(
        pre_trace_q, initial_truth_assignment
    ).next_state


def _format_counter(counter):
    """Convert an automaton transition counter into a readable string."""
    if not counter:
        return "none"
    return ", ".join(f"{source}->{destination}: {count}" for (source, destination), count in sorted(counter.items()))


# ==============================
# Logging and checkpoint helpers
# ==============================

def _write_log(message, log_handle=None):
    """Print a message and optionally append it to the active log file."""
    print(message)
    if log_handle:
        log_handle.write(message)
        log_handle.flush()


def _write_run_header(log_handle, episodes, use_shaping, K, goal_reward, abstract_mdp, automaton_states):
    """Write the configuration and automaton metadata for a training run."""
    if not log_handle:
        return
    automaton = abstract_mdp.automaton
    header = (
        "\n=== NEW RUN ===\n"
        f"episodes={episodes}, shaping={use_shaping}, K={K}, goal_reward={goal_reward}, gamma={abstract_mdp.gamma}\n"
        f"waypoints={abstract_mdp.waypoints_dict}\n"
        f"automaton_states={automaton_states}, initial={automaton.get_initial_q()}, accepting={sorted(automaton.accepting_states)}\n"
    )
    log_handle.write(header)
    log_handle.flush()


def _should_log(episode, episodes, log_interval):
    """Return whether the current episode requires a periodic training report."""
    return episode == 0 or episode + 1 == episodes or (episode + 1) % log_interval == 0


def _build_training_log(episode, episodes, log_interval, automaton_states, agent, histories, cumulative_counters):
    """Build a report containing recent metrics and automaton counters."""
    window = min(log_interval, episode + 1)
    recent_slice = slice(-window, None)
    recent_transitions = Counter()
    for transitions in histories["transition_counters"][-window:]:
        recent_transitions.update(transitions)

    recent_state_visits = np.asarray(histories["state_visits"], dtype=np.int64)[:, -window:].sum(axis=1)
    recent_state_entries = np.asarray(histories["state_entries"], dtype=np.int64)[:, -window:].sum(axis=1)
    buffer_details = ", ".join(f"{q}: {agent.memory.q_fraction_onehot(index, len(automaton_states)):.1%}" for index, q in enumerate(automaton_states))
    recent_visits_details = ", ".join(f"{q}: {recent_state_visits[index]}" for index, q in enumerate(automaton_states))
    recent_entries_details = ", ".join(f"{q}: {recent_state_entries[index]}" for index, q in enumerate(automaton_states))
    cumulative_visits_details = ", ".join(f"{q}: {cumulative_counters['state_visits'][q]}" for q in automaton_states)
    cumulative_entries_details = ", ".join(f"{q}: {cumulative_counters['state_entries'][q]}" for q in automaton_states)

    return (
        "\n"
        f"[Episode {episode + 1}/{episodes} | last {window}]\n"
        f"success rate                : {np.mean(histories['successes'][recent_slice]):.1%} (cumulative {np.mean(histories['successes']):.1%})\n"
        f"synthetic task reward       : {np.mean(histories['task_rewards'][recent_slice]):.3f}\n"
        f"shaping reward              : {np.mean(histories['shaping_rewards'][recent_slice]):.3f}\n"
        f"learning reward             : {np.mean(histories['learning_rewards'][recent_slice]):.3f}\n"
        f"episode length              : {np.mean(histories['episode_lengths'][recent_slice]):.1f}\n"
        f"abstract changes / episode  : {np.mean(histories['abstract_changes'][recent_slice]):.1f}\n"
        f"completed cycles / episode   : {np.mean(histories['completed_cycles'][recent_slice]):.2f}\n"
        f"automaton changes / episode  : {np.mean(histories['automaton_transitions'][recent_slice]):.2f}\n"
        f"automaton changes in window  : {_format_counter(recent_transitions)}\n"
        f"epsilon (next episode)       : {histories['epsilons'][-1]:.5f}\n"
        f"replay buffer                : {len(agent.memory)} samples [{buffer_details}]\n"
        f"state visits in window        : {recent_visits_details}\n"
        f"state visits cumulative       : {cumulative_visits_details}\n"
        f"state entries in window       : {recent_entries_details}\n"
        f"state entries cumulative      : {cumulative_entries_details}\n"
        f"transitions cumulative       : {_format_counter(cumulative_counters['transitions'])}\n"
        f"Gym endings cumulative       : terminated={cumulative_counters['env_terminated']}, truncated={cumulative_counters['env_truncated']}\n"
    )


def _save_named_policy(agent, policy_name):
    """Save the current policy using a stable descriptive filename."""
    agent.policy_name = policy_name
    agent._save_policy()


def _monitoring_average(values, episode, log_interval):
    """Return the mean over the active monitoring window."""
    window = min(log_interval, episode + 1)
    return float(np.mean(values[-window:]))


def _validate_training_setup(automaton, state_to_index, episodes, log_interval):
    """Validate automaton consistency and numeric training parameters."""
    if automaton.get_initial_q() not in state_to_index:
        raise ValueError("The initial state is missing from automaton.states")
    if set(state_to_index) != set(automaton.active_states):
        raise ValueError("Network phases must match the stable automaton states")
    if episodes <= 0:
        raise ValueError("episodes must be greater than zero")
    if log_interval <= 0:
        raise ValueError("log_interval must be greater than zero")


def _build_training_results(histories, buffer_histories, automaton_states, best_mean_reward, best_policy_episode):
    """Select and name the numeric histories returned by the training loop."""
    return {
        "task_rewards": histories["task_rewards"],
        "learning_rewards": histories["learning_rewards"],
        "shaping_rewards": histories["shaping_rewards"],
        "epsilon_history": histories["epsilons"],
        "buffer_histories": buffer_histories,
        "state_visit_histories": histories["state_visits"],
        "state_entry_histories": histories["state_entries"],
        "successes": histories["successes"],
        "completed_cycles": histories["completed_cycles"],
        "episode_lengths": histories["episode_lengths"],
        "abstract_changes": histories["abstract_changes"],
        "automaton_transitions": histories["automaton_transitions"],
        "automaton_states": automaton_states,
        "best_mean_learning_reward": best_mean_reward,
        "best_policy_episode": best_policy_episode,
    }


# ==============================
# Training loop
# ==============================

def run_sequential_training(env, agent, abstract_mdp, episodes, goal_reward=10000, save_policy=True, use_shaping=True, K=1.0, log_file=None, log_interval=100):
    """
    Train the DDQN agent with the manual automaton and one global epsilon.

    The Gym reward is deliberately discarded. The learning reward is the
    synthetic goal reward plus potential-based shaping. Shaping is evaluated
    only when the complete abstract state (x, y, q) changes.
    """
    # Build a stable mapping between automaton states and network features.
    automaton = abstract_mdp.automaton
    automaton_states = list(automaton.active_states)
    state_to_index = {q: index for index, q in enumerate(automaton_states)}
    num_states = len(automaton_states)

    # Fail early if the automaton or training parameters are inconsistent.
    _validate_training_setup(automaton, state_to_index, episodes, log_interval)

    # Store episode-level metrics for plots and post-processing.
    task_reward_history = []
    learning_reward_history = []
    shaping_reward_history = []
    epsilon_history = []
    episode_length_history = []
    success_history = []
    completed_cycle_history = []
    abstract_change_history = []
    automaton_transition_history = []
    transition_counter_history = []
    buffer_histories = [[] for _ in automaton_states]
    state_visit_histories = [[] for _ in automaton_states]
    state_entry_histories = [[] for _ in automaton_states]
    histories = {
        "task_rewards": task_reward_history,
        "learning_rewards": learning_reward_history,
        "shaping_rewards": shaping_reward_history,
        "epsilons": epsilon_history,
        "episode_lengths": episode_length_history,
        "successes": success_history,
        "completed_cycles": completed_cycle_history,
        "abstract_changes": abstract_change_history,
        "automaton_transitions": automaton_transition_history,
        "transition_counters": transition_counter_history,
        "state_visits": state_visit_histories,
        "state_entries": state_entry_histories,
    }

    # Keep cumulative counters for diagnostics shown during training.
    cumulative_state_visits = Counter()
    cumulative_state_entries = Counter()
    cumulative_transitions = Counter()
    cumulative_env_terminated = 0
    cumulative_env_truncated = 0
    best_mean_reward = -np.inf
    best_policy_episode = 0

    # Open one append-only log file for the complete run.
    log_handle = open(log_file, "a", encoding="utf-8") if log_file else None
    _write_run_header(log_handle, episodes, use_shaping, K, goal_reward, abstract_mdp, automaton_states)

    try:
        for episode in range(episodes):
            # Reset the environment and consume s0 before selecting an action.
            raw_state, _ = env.reset()
            q = _evaluate_initial_automaton_state(raw_state, abstract_mdp)
            if q not in state_to_index:
                raise RuntimeError(f"Automaton returned unknown initial state {q!r}")
            augmented_state = _augment_state(raw_state, q, state_to_index)

            # Reset counters local to the current episode.
            succeeded = False
            episode_done = False
            episode_steps = 0
            episode_task_reward = 0.0
            episode_shaping_reward = 0.0
            episode_abstract_changes = 0
            episode_automaton_transitions = 0
            episode_completed_cycles = 0
            episode_state_visits = [0] * num_states
            episode_state_visits[state_to_index[q]] = 1
            # Count s0 as an entry from the virtual pre-episode state.
            episode_state_entries = [0] * num_states
            episode_state_entries[state_to_index[q]] = 1
            episode_transitions = Counter()
            cumulative_state_visits[q] += 1
            cumulative_state_entries[q] += 1

            while not episode_done:
                # Select an action using the single global epsilon.
                agent.eps = epsilon_history[-1] if epsilon_history else agent.eps
                action = agent.select_action(augmented_state)

                # The environment reward is intentionally not part of training.
                next_raw_state, _ignored_env_reward, env_terminated, env_truncated, _ = env.step(action)

                # Map the transition to abstract spatial states.
                x, y = _abstract_position(raw_state, abstract_mdp)
                next_x, next_y = _abstract_position(next_raw_state, abstract_mdp)
                abstract_state = (x, y, q)

                # Advance the automaton using propositions true on arrival.
                truth_assignment = abstract_mdp._get_truth_assignment(next_x, next_y)
                automaton_step = automaton.advance(q, truth_assignment)
                next_q = automaton_step.next_state
                if next_q not in state_to_index:
                    raise RuntimeError(f"Automaton returned unknown state {next_q!r} from state {q!r}")

                # Count every arrival in an automaton state, including self-loops.
                episode_state_visits[state_to_index[next_q]] += 1
                cumulative_state_visits[next_q] += 1

                # Track physical abstraction and automaton changes separately.
                abstract_next_state = (next_x, next_y, next_q)
                abstract_changed = abstract_state != abstract_next_state
                automaton_changed = next_q != q

                if abstract_changed:
                    episode_abstract_changes += 1
                if automaton_changed:
                    transition = (q, next_q)
                    episode_automaton_transitions += 1
                    episode_state_entries[state_to_index[next_q]] += 1
                    episode_transitions[transition] += 1
                    cumulative_state_entries[next_q] += 1
                    cumulative_transitions[transition] += 1

                # Reward the final waypoint once per completed cycle.
                completed_cycle = automaton_step.completed_cycle
                synthetic_goal_reward = (
                    float(goal_reward) if completed_cycle else 0.0
                )
                if completed_cycle:
                    succeeded = True
                    episode_completed_cycles += 1

                # Acceptance does not end an episode. Only Gymnasium can do so.
                # A truncation (for example Gym's time limit) ends data
                # collection, but it is not an MDP terminal state: DDQN must
                # still bootstrap from its final observation.
                episode_done = env_terminated or env_truncated
                bootstrap_terminal = env_terminated
                next_augmented_state = _augment_state(next_raw_state, next_q, state_to_index)

                # Evaluate shaping only when the complete abstract state changes.
                shaping_signal = 0.0
                if use_shaping and abstract_changed:
                    phi_state = abstract_mdp.v_star.get(abstract_state, 0.0)
                    phi_next_state = abstract_mdp.v_star.get(abstract_next_state, 0.0)
                    shaping_signal = K * (abstract_mdp.gamma * phi_next_state - phi_state)

                # Store the transition and perform one DDQN optimization step.
                learning_reward = synthetic_goal_reward + shaping_signal
                agent.memory.push(
                    augmented_state,
                    action,
                    learning_reward,
                    next_augmented_state,
                    bootstrap_terminal,
                )
                agent.optimize_model()

                # Update the episode totals and move to the next state.
                episode_steps += 1
                episode_task_reward += synthetic_goal_reward
                episode_shaping_reward += shaping_signal
                raw_state = next_raw_state
                augmented_state = next_augmented_state
                q = next_q

                # Count Gym endings for diagnostics without using its reward.
                if env_terminated:
                    cumulative_env_terminated += 1
                if env_truncated:
                    cumulative_env_truncated += 1

            # Decay the single epsilon once at the end of the episode.
            next_epsilon = max(agent.eps_min, agent.eps * agent.eps_decay)
            agent.eps = next_epsilon

            # Save the metrics collected for this episode.
            episode_learning_reward = episode_task_reward + episode_shaping_reward
            task_reward_history.append(episode_task_reward)
            shaping_reward_history.append(episode_shaping_reward)
            learning_reward_history.append(episode_learning_reward)
            epsilon_history.append(next_epsilon)
            episode_length_history.append(episode_steps)
            success_history.append(int(succeeded))
            completed_cycle_history.append(episode_completed_cycles)
            abstract_change_history.append(episode_abstract_changes)
            automaton_transition_history.append(episode_automaton_transitions)
            transition_counter_history.append(episode_transitions)

            # Record replay-buffer composition, state visits, and entries from other states.
            for index in range(num_states):
                buffer_histories[index].append(agent.memory.q_fraction_onehot(index, num_states))
                state_visit_histories[index].append(episode_state_visits[index])
                state_entry_histories[index].append(episode_state_entries[index])

            # Print recent and cumulative diagnostics at the requested interval.
            if _should_log(episode, episodes, log_interval):
                cumulative_counters = {"state_visits": cumulative_state_visits, "state_entries": cumulative_state_entries, "transitions": cumulative_transitions, "env_terminated": cumulative_env_terminated, "env_truncated": cumulative_env_truncated}
                _write_log(_build_training_log(episode, episodes, log_interval, automaton_states, agent, histories, cumulative_counters), log_handle)

                # Replace the best policy when the monitored mean reward improves.
                monitored_mean_reward = _monitoring_average(learning_reward_history, episode, log_interval)
                if monitored_mean_reward > best_mean_reward:
                    best_mean_reward = monitored_mean_reward
                    best_policy_episode = episode + 1
                    if save_policy:
                        _save_named_policy(agent, "best_policy.pth")
                        _write_log(f"Best policy updated at episode {best_policy_episode}: mean learning reward={best_mean_reward:.3f}\n", log_handle)

        # Save the final policy independently from its monitored performance.
        if save_policy:
            _save_named_policy(agent, "last_policy.pth")
            _write_log(f"Last policy saved after episode {episodes}. Best policy: episode {best_policy_episode}, mean learning reward={best_mean_reward:.3f}\n", log_handle)
    finally:
        # Always close the log, including when training raises an exception.
        if log_handle:
            log_handle.close()

    # Return named histories to avoid ambiguous tuple positions.
    return _build_training_results(histories, buffer_histories, automaton_states, best_mean_reward, best_policy_episode)


# ==============================
# Experiment setup and outputs
# ==============================

def main(args):
    """Configure the experiment, run or load training, and generate diagnostic plots."""
    # Prepare output directories shared by training and post-processing.
    data_dir = "results"
    image_dir = "img"
    log_dir = "logs"
    for directory in (data_dir, image_dir, log_dir):
        os.makedirs(directory, exist_ok=True)
    plot_dir = data_dir if args.post_process else image_dir

    # Load the manual task and optional training parameters.
    with open(args.config, "r", encoding="utf-8") as config_file:
        config = json.load(config_file)

    waypoints = {
        name: tuple(coordinates)
        for name, coordinates in config.get(
            "waypoints_dict", {"g1": [1, 8], "g2": [8, 8]}
        ).items()
    }
    gamma = float(config.get("gamma", 0.99))
    goal_reward = float(config.get("goal_reward", 10000))
    grid_w = int(config.get("grid_w", 12))
    grid_h = int(config.get("grid_h", 12))

    # The cycle order is explicit; when omitted, JSON waypoint insertion order
    # provides a convenient backwards-compatible default.
    waypoint_cycle = config.get("waypoint_cycle", list(waypoints))
    automaton = CyclicWaypointsAutomaton(waypoint_cycle)
    automaton.validate_waypoints(waypoints, width=grid_w, height=grid_h)
    print(
        "=== MANUAL AUTOMATON TRAINING (single epsilon) ===\n"
        f"Waypoints: {waypoints}\n"
        f"Automaton: states={automaton.states}, stable={automaton.active_states}, "
        f"initial={automaton.initial_state}, "
        f"accepting={sorted(automaton.accepting_states)}\n"
        f"Cycle: {automaton.describe_cycle()}\n"
        "Gym reward is ignored; acceptance does not end an episode."
    )

    if not args.post_process:
        # Create the environment and abstract MDP used to compute the potential.
        automaton.render_graph()
        env = gym.make("LunarLander-v3", continuous=False)
        try:
            abstract_mdp = ManualWaypointMDP(
                waypoints_dict=waypoints,
                automaton=automaton,
                width=grid_w,
                height=grid_h,
                gamma=gamma,
                goal_reward=goal_reward,
            )
            abstract_mdp.value_iteration()
            save_sequential_heatmaps(abstract_mdp, filename_prefix="single_epsilon_exp")

            # Initialize one DDQN agent with a single exploration schedule.
            agent = HierarchicalDQNLearner(
                env=env,
                max_episodes=args.episodes,
                eps_decay=args.eps_decay,
                gamma=gamma,
                extra_state_dims=len(automaton.active_states),
                use_polyak=args.polyak,
                tau=args.polyak_tau,
                target_update_freq=args.target_update_freq,
                network_type=args.network_type,
            )

            # Run training and persist all collected metrics.
            metrics = run_sequential_training(env=env, agent=agent, abstract_mdp=abstract_mdp, episodes=args.episodes, goal_reward=goal_reward, use_shaping=not args.no_shaping, K=args.shaping_scale, log_file=f"{log_dir}/single_epsilon_training.log", log_interval=args.log_interval)
            save_training_data(f"{data_dir}/single_epsilon_data.npz", **metrics)
        finally:
            # Release environment resources even if training fails.
            env.close()

    # Load saved metrics and generate the final diagnostic plots.
    data = np.load(f"{data_dir}/single_epsilon_data.npz", allow_pickle=False)
    plot_buffer_fractions(data["buffer_histories"], filename=f"{plot_dir}/buffer_fractions_single_epsilon.png", window_size=args.plot_window, state_labels=data["automaton_states"])
    plot_shaping_reward_breakdown(data["task_rewards"], data["learning_rewards"], data["epsilon_history"], window_size=args.plot_window, filename=f"{plot_dir}/reward_breakdown_single_epsilon.png")
    print("\nFinished.")


# ==============================
# Command-line entry point
# ==============================

if __name__ == "__main__":
    # Expose the main training and post-processing options.
    parser = argparse.ArgumentParser(description="Manual-automaton DDQN training with one global epsilon.")
    parser.add_argument("--episodes", type=int, default=1000)
    parser.add_argument("--config", default="trajectory.json")
    parser.add_argument("--eps-decay", type=float, default=0.9996)
    parser.add_argument("--shaping-scale", type=float, default=1.0)
    parser.add_argument("--log-interval", type=int, default=100)
    parser.add_argument("--plot-window", type=int, default=500)
    parser.add_argument(
        "--polyak",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="Use Polyak target updates (disable with --no-polyak).",
    )
    parser.add_argument("--polyak-tau", type=float, default=0.005)
    parser.add_argument(
        "--target-update-freq",
        type=int,
        default=1000,
        help="Hard target-network update interval used with --no-polyak.",
    )
    parser.add_argument(
        "--network-type",
        choices=["standard", "dueling"],
        default="standard",
        help="Q-network architecture: standard MLP or dueling value/advantage streams.",
    )
    parser.add_argument("--no-shaping", action="store_true")
    parser.add_argument("--post-process", action="store_true")
    main(parser.parse_args())


## 9. Write grid visualization and evaluation modules


In [ ]:
%%writefile grid_overlay.py
"""Visualise the abstract grid on top of the LunarLander environment.

The conversion used here is the inverse of ``utils.phi_mapping_grid``.  Grid
cells in the resulting image therefore represent exactly the abstract states
used by the trainer, rather than an evenly spaced, screen-only decoration.
"""

from __future__ import annotations

import argparse
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Mapping, Sequence

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

from utils import phi_mapping_grid, spatial_grid_boundaries


SCRIPT_DIR = Path(__file__).resolve().parent
DEFAULT_CONFIG = SCRIPT_DIR / "trajectory.json"


@dataclass(frozen=True)
class LunarLanderGeometry:
    """Screen/world constants required to project observations onto a frame."""

    viewport_width: int
    viewport_height: int
    scale: float
    helipad_y: float
    leg_down: float


def geometry_from_env(env: gym.Env) -> LunarLanderGeometry:
    """Read projection constants from a reset LunarLander environment."""
    from gymnasium.envs.box2d import lunar_lander

    base_env = env.unwrapped
    if not hasattr(base_env, "helipad_y"):
        raise TypeError("The supplied environment is not a LunarLander environment")
    return LunarLanderGeometry(
        viewport_width=lunar_lander.VIEWPORT_W,
        viewport_height=lunar_lander.VIEWPORT_H,
        scale=lunar_lander.SCALE,
        helipad_y=float(base_env.helipad_y),
        leg_down=lunar_lander.LEG_DOWN,
    )


def observation_to_pixel(
    observation: Sequence[float], geometry: LunarLanderGeometry
) -> tuple[float, float]:
    """Project LunarLander's normalised (x, y) observation onto RGB pixels."""
    half_world_width = geometry.viewport_width / geometry.scale / 2.0
    half_world_height = geometry.viewport_height / geometry.scale / 2.0
    world_x = (float(observation[0]) + 1.0) * half_world_width
    world_y = (
        float(observation[1]) * half_world_height
        + geometry.helipad_y
        + geometry.leg_down / geometry.scale
    )
    pixel_x = world_x * geometry.scale
    pixel_y = geometry.viewport_height - world_y * geometry.scale
    return pixel_x, pixel_y


def pixel_to_observation(
    pixel_x: float,
    pixel_y: float,
    geometry: LunarLanderGeometry,
) -> tuple[float, float]:
    """Invert the frame projection for the two discretised coordinates."""
    half_world_width = geometry.viewport_width / geometry.scale / 2.0
    half_world_height = geometry.viewport_height / geometry.scale / 2.0
    world_x = float(pixel_x) / geometry.scale
    world_y = (geometry.viewport_height - float(pixel_y)) / geometry.scale
    observation_x = world_x / half_world_width - 1.0
    observation_y = (
        world_y - geometry.helipad_y - geometry.leg_down / geometry.scale
    ) / half_world_height
    return observation_x, observation_y


def _grid_boundaries(
    grid_w: int,
    grid_h: int,
    geometry: LunarLanderGeometry,
) -> tuple[np.ndarray, np.ndarray]:
    """Return the boundaries implied by the active spatial discretizer."""
    x_normalised, y_normalised = spatial_grid_boundaries(grid_w, grid_h)
    x_pixels = np.array(
        [observation_to_pixel((x, 0.0), geometry)[0] for x in x_normalised]
    )
    y_pixels = np.array(
        [observation_to_pixel((0.0, y), geometry)[1] for y in y_normalised]
    )

    # phi_mapping_grid clips observations outside its nominal domain into its
    # edge cells. Extend those cells to the RGB viewport edges whenever the
    # corresponding border observation maps to index 0 or to the last index.
    # Internal boundaries remain entirely inferred from the active mapper.
    left_observation_x, top_observation_y = pixel_to_observation(
        0.0, 0.0, geometry
    )
    right_observation_x, bottom_observation_y = pixel_to_observation(
        geometry.viewport_width, geometry.viewport_height, geometry
    )
    if phi_mapping_grid((left_observation_x, 0.0), grid_w, grid_h)[0] == 0:
        x_pixels[0] = min(x_pixels[0], 0.0)
    if phi_mapping_grid((right_observation_x, 0.0), grid_w, grid_h)[0] == grid_w - 1:
        x_pixels[-1] = max(x_pixels[-1], float(geometry.viewport_width))
    if phi_mapping_grid((0.0, bottom_observation_y), grid_w, grid_h)[1] == 0:
        y_pixels[0] = max(y_pixels[0], float(geometry.viewport_height))
    if phi_mapping_grid((0.0, top_observation_y), grid_w, grid_h)[1] == grid_h - 1:
        y_pixels[-1] = min(y_pixels[-1], 0.0)

    return x_pixels, y_pixels


def abstract_cell_to_pixel(
    grid_x: int,
    grid_y: int,
    grid_w: int,
    grid_h: int,
    geometry: LunarLanderGeometry,
) -> tuple[float, float]:
    """Return the pixel coordinates of an abstract cell's centre."""
    if not (0 <= grid_x < grid_w and 0 <= grid_y < grid_h):
        raise ValueError(f"Abstract cell ({grid_x}, {grid_y}) is outside the grid")
    x_lines, y_lines = _grid_boundaries(grid_w, grid_h, geometry)
    return (
        float((x_lines[grid_x] + x_lines[grid_x + 1]) / 2.0),
        float((y_lines[grid_y] + y_lines[grid_y + 1]) / 2.0),
    )


def draw_abstract_grid(
    frame: np.ndarray,
    geometry: LunarLanderGeometry,
    grid_w: int,
    grid_h: int,
    waypoints: Mapping[str, Sequence[int]] | None = None,
    observation: Sequence[float] | None = None,
    title: str = "LunarLander with Abstract Grid",
):
    """Create a figure containing the effective clipped grid and its markers."""
    if grid_w < 2 or grid_h < 2:
        raise ValueError("grid_w and grid_h must both be at least 2")

    figure, axis = plt.subplots(figsize=(12, 8))
    axis.imshow(frame)
    x_lines, y_lines = _grid_boundaries(grid_w, grid_h, geometry)
    x_centres = (x_lines[:-1] + x_lines[1:]) / 2.0
    y_centres = (y_lines[:-1] + y_lines[1:]) / 2.0
    grid_color = "#ff1744"

    for x_pixel in x_lines:
        axis.axvline(x_pixel, color=grid_color, linewidth=1.6, alpha=0.95)
    for y_pixel in y_lines:
        axis.axhline(y_pixel, color=grid_color, linewidth=1.6, alpha=0.95)

    # The mapping clips everything outside its stated observation domain into
    # an edge cell; tint the currently occupied abstract cell when requested.
    if observation is not None:
        abstract_x, abstract_y = phi_mapping_grid(observation, grid_w, grid_h)
        x0, x1 = sorted((x_lines[abstract_x], x_lines[abstract_x + 1]))
        y0, y1 = sorted((y_lines[abstract_y], y_lines[abstract_y + 1]))
        x0, x1 = np.clip((x0, x1), 0, geometry.viewport_width)
        y0, y1 = np.clip((y0, y1), 0, geometry.viewport_height)
        axis.add_patch(
            Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                facecolor="#00e5ff",
                edgecolor="#00e5ff",
                linewidth=2.5,
                alpha=0.25,
                label=f"Current cell ({abstract_x}, {abstract_y})",
            )
        )

    for name, coordinates in (waypoints or {}).items():
        if len(coordinates) != 2:
            raise ValueError(f"Waypoint {name!r} must contain [x, y]")
        grid_x, grid_y = int(coordinates[0]), int(coordinates[1])
        if not (0 <= grid_x < grid_w and 0 <= grid_y < grid_h):
            raise ValueError(f"Waypoint {name!r} is outside the abstract grid")
        pixel_x, pixel_y = abstract_cell_to_pixel(
            grid_x, grid_y, grid_w, grid_h, geometry
        )
        axis.scatter(pixel_x, pixel_y, s=150, marker="o", color="#ffca28",
                     edgecolor="black", linewidth=1.3, zorder=5)
        axis.annotate(
            f"{name} ({grid_x}, {grid_y})",
            (pixel_x, pixel_y),
            xytext=(7, -10),
            textcoords="offset points",
            color="black",
            fontsize=9,
            fontweight="bold",
            bbox={"boxstyle": "round,pad=0.25", "fc": "#ffca28", "alpha": 0.9},
            zorder=6,
        )

    axis.set_xlim(0, geometry.viewport_width)
    # A small part of the configured y-domain can lie above the RGB viewport.
    # Keep it in view so that no abstract row or coordinate label disappears.
    visible_top = min(0.0, float(np.min(y_lines)))
    axis.set_ylim(geometry.viewport_height, visible_top)
    axis.set_title(title)

    # Put the abstract coordinates at cell centres. Since image coordinates
    # grow downwards while abstract y grows upwards, y_centres is descending:
    # label 0 consequently appears at the bottom and grid_h - 1 at the top.
    axis.set_xticks(x_centres, labels=range(grid_w))
    axis.set_yticks(y_centres, labels=range(grid_h))
    axis.set_xlabel("Abstract x-coordinate")
    axis.set_ylabel("Abstract y-coordinate")
    axis.tick_params(
        axis="both",
        which="major",
        color=grid_color,
        labelcolor=grid_color,
        labelsize=10,
        width=1.5,
        length=5,
    )
    for label in (*axis.get_xticklabels(), *axis.get_yticklabels()):
        label.set_fontweight("bold")

    if observation is not None:
        axis.legend(
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
            frameon=True,
        )
        figure.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
    else:
        figure.tight_layout()
    return figure


def generate_overlay(
    output_path: str | Path,
    config_path: str | Path = DEFAULT_CONFIG,
    seed: int | None = 0,
) -> Path:
    """Reset LunarLander and save one annotated RGB frame as a PNG."""
    config_path = Path(config_path)
    with config_path.open(encoding="utf-8") as config_file:
        config = json.load(config_file)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    env = gym.make("LunarLander-v3", continuous=False, render_mode="rgb_array")
    try:
        observation, _ = env.reset(seed=seed)
        frame = env.render()
        geometry = geometry_from_env(env)
        figure = draw_abstract_grid(
            frame=frame,
            geometry=geometry,
            grid_w=int(config.get("grid_w", 12)),
            grid_h=int(config.get("grid_h", 12)),
            waypoints=config.get("waypoints_dict", {}),
            observation=observation,
        )
        figure.savefig(output_path, dpi=180, bbox_inches="tight")
        plt.close(figure)
    finally:
        env.close()
    return output_path.resolve()


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Generate a LunarLander frame with the abstract grid overlaid."
    )
    parser.add_argument("--config", type=Path, default=DEFAULT_CONFIG)
    parser.add_argument("--output", type=Path)
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()
    output_path = args.output or SCRIPT_DIR / "img" / "abstract_grid_overlay.png"
    saved_path = generate_overlay(output_path, args.config, args.seed)
    print(f"Image saved to: {saved_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
"""Evaluate policies trained with the manual cyclic-waypoints automaton."""

# ==============================
# Standard library imports
# ==============================

import argparse
import json
import time
from pathlib import Path

# ==============================
# External and project imports
# ==============================

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch

from abstract_mdps import ManualWaypointMDP
from agent import DuelingQNetwork, QNetwork
from grid_overlay import (
    abstract_cell_to_pixel,
    draw_abstract_grid,
    geometry_from_env,
)
from manual_automaton import CyclicWaypointsAutomaton
from utils import phi_mapping_sequential


# ==============================
# Paths and generic helpers
# ==============================

SCRIPT_DIR = Path(__file__).resolve().parent
EXPERIMENTS_DIR = SCRIPT_DIR.parent / "experiments"


def moving_average(data, window_size):
    """Return a moving average, or the original values when the window is larger."""
    values = np.asarray(data, dtype=np.float64)
    if len(values) < window_size:
        return values
    return np.convolve(values, np.ones(window_size) / window_size, mode="valid")


def _resolve_policy_path(policy, policy_dir):
    """Accept explicit paths as well as filenames relative to the policy directory."""
    supplied_path = Path(policy).expanduser()
    if supplied_path.is_file():
        return supplied_path.resolve()

    policy_path = Path(policy_dir).expanduser() / supplied_path
    if policy_path.is_file():
        return policy_path.resolve()

    raise FileNotFoundError(f"Policy '{policy}' not found either as an explicit path or under '{policy_dir}'.")


def _load_state_dict(policy_path, device):
    """Load both plain state dictionaries and common wrapped checkpoints."""
    checkpoint = torch.load(policy_path, map_location=device, weights_only=True)
    if isinstance(checkpoint, dict):
        for key in ("policy_state_dict", "state_dict", "model_state_dict"):
            if key in checkpoint:
                return checkpoint[key]
    return checkpoint


def _abstract_position(observation, q, grid_w, grid_h):
    """Map an environment observation to its abstract grid coordinates."""
    x, y, _ = phi_mapping_sequential(observation, q, grid_w, grid_h)
    return x, y


# ==============================
# Policy evaluation
# ==============================

def evaluate_policy(policy, policy_dir, episodes, render, waypoints_dict, goal_reward, grid_w, grid_h, seed, trace_episodes=0, network_type="standard", waypoint_cycle=None, no_limit=False):
    """Load and evaluate one policy using the training automaton semantics."""
    # Rebuild the same automaton and abstract MDP used during training.
    policy_path = _resolve_policy_path(policy, policy_dir)
    policy_name = policy_path.name
    cycle = list(waypoints_dict) if waypoint_cycle is None else waypoint_cycle
    automaton = CyclicWaypointsAutomaton(cycle)
    automaton.validate_waypoints(waypoints_dict, width=grid_w, height=grid_h)
    abstract_mdp = ManualWaypointMDP(
        waypoints_dict=waypoints_dict,
        automaton=automaton,
        width=grid_w,
        height=grid_h,
        goal_reward=goal_reward,
    )
    automaton_states = list(automaton.active_states)
    state_to_index = {q: index for index, q in enumerate(automaton_states)}

    # Create the environment and one extra network feature per automaton state.
    render_mode = "human" if render else ("rgb_array" if trace_episodes else None)
    environment_options = {"continuous": False, "render_mode": render_mode}
    if no_limit:
        environment_options["max_episode_steps"] = 5000
    env = gym.make("LunarLander-v3", **environment_options)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if network_type not in {"standard", "dueling"}:
        raise ValueError("network_type must be one of: standard, dueling")
    network_cls = DuelingQNetwork if network_type == "dueling" else QNetwork
    network = network_cls(env.observation_space.shape[0] + len(automaton_states), env.action_space.n).to(device)

    # Load the trained parameters before starting any episode.
    try:
        network.load_state_dict(_load_state_dict(policy_path, device))
        network.eval()
    except Exception:
        env.close()
        raise

    task_returns = []
    environment_returns = []
    episode_lengths = []
    successes = 0
    completed_cycles = []
    state_reach_counts = {q: 0 for q in automaton_states}
    grid_traces = []
    trace_frames = []
    trace_geometries = []

    # Run every requested episode sequentially.
    try:
        for episode in range(episodes):
            episode_seed = None if seed is None else seed + episode
            observation, _ = env.reset(seed=episode_seed)
            tracing = episode < trace_episodes
            if tracing:
                trace_frames.append(env.render())
                trace_geometries.append(geometry_from_env(env))
                initial_cell = _abstract_position(observation, automaton.get_initial_q(), grid_w, grid_h)
                cell_trace = [initial_cell]

            # Training consumes the valuation at s0 before choosing an action.
            initial_q = automaton.get_initial_q()
            initial_x, initial_y = _abstract_position(observation, initial_q, grid_w, grid_h)
            initial_truth_assignment = abstract_mdp._get_truth_assignment(initial_x, initial_y)
            q = automaton.advance(
                initial_q, initial_truth_assignment
            ).next_state
            if q not in state_to_index:
                raise RuntimeError(f"Automaton returned unknown state {q!r}")

            reached_states = {q}
            episode_completed_cycles = 0
            terminated = truncated = False
            environment_return = 0.0
            steps = 0

            while not (terminated or truncated):
                # Append the current automaton state as a one-hot vector.
                one_hot = np.zeros(len(automaton_states), dtype=np.float32)
                one_hot[state_to_index[q]] = 1.0
                augmented_state = np.concatenate((observation, one_hot)).astype(np.float32)

                # Evaluation is greedy: always select the action with maximum Q-value.
                with torch.inference_mode():
                    state_tensor = torch.as_tensor(augmented_state, device=device).unsqueeze(0)
                    action = network(state_tensor).argmax(dim=1).item()

                next_observation, env_reward, terminated, truncated, _ = env.step(action)
                environment_return += float(env_reward)
                steps += 1

                # Advance the automaton using propositions true on arrival.
                x, y = _abstract_position(next_observation, q, grid_w, grid_h)
                if tracing and (x, y) != cell_trace[-1]:
                    cell_trace.append((x, y))
                truth_assignment = abstract_mdp._get_truth_assignment(x, y)
                automaton_step = automaton.advance(q, truth_assignment)
                next_q = automaton_step.next_state
                if next_q not in state_to_index:
                    raise RuntimeError(f"Automaton returned unknown state {next_q!r}")

                completed_cycle = automaton_step.completed_cycle
                if completed_cycle:
                    episode_completed_cycles += 1

                # Report every consumed waypoint, including a one-waypoint cycle
                # whose epsilon reset leaves the same stable state active.
                if automaton_step.reached_waypoint is not None:
                    transition = f"{q} -> {next_q}"
                    if completed_cycle:
                        transition = f"{q} -> {automaton.accepting_state} -> {next_q}"
                    suffix = (
                        f", cycle {episode_completed_cycles} completed with "
                        "immediate epsilon reset"
                        if completed_cycle
                        else ""
                    )
                    print(
                        f"[{policy_name} | Episode {episode + 1}] "
                        f"automaton transition {transition}: "
                        f"{automaton_step.reached_waypoint} reached{suffix}."
                    )

                reached_states.add(next_q)

                observation = next_observation
                q = next_q

                if render:
                    time.sleep(0.02)

            # Store metrics and count every state reached at least once.
            success = episode_completed_cycles > 0
            successes += int(success)
            completed_cycles.append(episode_completed_cycles)
            for reached_q in reached_states:
                state_reach_counts[reached_q] += 1
            task_returns.append(float(goal_reward) * episode_completed_cycles)
            environment_returns.append(environment_return)
            episode_lengths.append(steps)
            if tracing:
                grid_traces.append(cell_trace)
    finally:
        env.close()

    return {
        "policy": policy_name,
        "path": str(policy_path),
        "task_returns": task_returns,
        "environment_returns": environment_returns,
        "episode_lengths": episode_lengths,
        "successes": successes,
        "completed_cycles": completed_cycles,
        "state_reach_counts": state_reach_counts,
        "grid_traces": grid_traces,
        "trace_frames": trace_frames,
        "trace_geometries": trace_geometries,
    }


# ==============================
# Plotting helpers
# ==============================

def _safe_stem(name):
    """Create a filesystem-safe plot stem from a checkpoint filename."""
    return "".join(character if character.isalnum() or character in "-_." else "_" for character in Path(name).stem)


def plot_policy(result, window_size, output_dir):
    """Plot Gym returns for one policy."""
    returns = result["environment_returns"]
    smooth = moving_average(returns, window_size)

    plt.figure(figsize=(10, 6))
    plt.plot(returns, alpha=0.3, color="gray", label="Raw Gym return")
    start = window_size - 1 if len(returns) >= window_size else 0
    plt.plot(range(start, start + len(smooth)), smooth, color="blue", linewidth=2, label=f"Moving average (window={window_size})")
    plt.title(f"Evaluation: {result['policy']}")
    plt.xlabel("Episode")
    plt.ylabel("Gym return")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    output_path = output_dir / f"eval_{_safe_stem(result['policy'])}.png"
    plt.savefig(output_path, bbox_inches="tight")
    plt.close()
    return output_path


def plot_comparison(results, window_size, output_dir):
    """Plot smoothed Gym returns for multiple policies."""
    plt.figure(figsize=(12, 6))
    for result in results:
        returns = result["environment_returns"]
        smooth = moving_average(returns, window_size)
        start = window_size - 1 if len(returns) >= window_size else 0
        plt.plot(range(start, start + len(smooth)), smooth, linewidth=2, label=result["policy"])
    plt.title("Policy comparison")
    plt.xlabel("Episode")
    plt.ylabel("Gym return")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    output_path = output_dir / "policy_comparison.png"
    plt.savefig(output_path, bbox_inches="tight")
    plt.close()
    return output_path


def plot_grid_traces(result, waypoints_dict, grid_w, grid_h, output_dir):
    """Save one abstract-grid path image for every recorded episode."""
    output_paths = []
    trace_data = zip(
        result["grid_traces"],
        result["trace_frames"],
        result["trace_geometries"],
    )
    for episode_index, (cells, frame, geometry) in enumerate(trace_data, start=1):
        figure = draw_abstract_grid(
            frame=frame,
            geometry=geometry,
            grid_w=grid_w,
            grid_h=grid_h,
            waypoints=waypoints_dict,
            title=f"Agent Abstract-Cell Trace — Episode {episode_index}",
        )
        axis = figure.axes[0]
        points = [
            abstract_cell_to_pixel(x, y, grid_w, grid_h, geometry)
            for x, y in cells
        ]
        if points:
            pixel_x, pixel_y = zip(*points)
            axis.plot(
                pixel_x,
                pixel_y,
                color="#00e5ff",
                linewidth=2.8,
                marker="o",
                markersize=5,
                label="Visited-cell path",
                zorder=4,
            )
            for change_index, ((cell_x, cell_y), (point_x, point_y)) in enumerate(
                zip(cells, points)
            ):
                axis.annotate(
                    str(change_index),
                    (point_x, point_y),
                    ha="center",
                    va="center",
                    fontsize=7,
                    fontweight="bold",
                    color="black",
                    zorder=7,
                )
        axis.legend(
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
            frameon=True,
        )
        figure.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
        output_path = output_dir / (
            f"grid_trace_{_safe_stem(result['policy'])}_episode_{episode_index}.png"
        )
        figure.savefig(output_path, dpi=180, bbox_inches="tight")
        plt.close(figure)
        output_paths.append(output_path)
    return output_paths


def format_waypoint_trace(cells, waypoints_dict):
    """Report the first cell-change index at which each waypoint was visited."""
    first_visit = {}
    for index, cell in enumerate(cells):
        first_visit.setdefault(tuple(cell), index)
    return ", ".join(
        f"{name}=reached@{first_visit[tuple(position)]}"
        if tuple(position) in first_visit
        else f"{name}=missed"
        for name, position in waypoints_dict.items()
    )


# ==============================
# Command-line interface
# ==============================

def _positive_int(value):
    """Parse and validate a strictly positive integer."""
    parsed = int(value)
    if parsed <= 0:
        raise argparse.ArgumentTypeError("must be greater than zero")
    return parsed


def _select_files_graphically(policy_dir, config_path):
    """Select policy checkpoints and the experiment configuration with native dialogs."""
    try:
        import tkinter as tk
        from tkinter import filedialog
    except ImportError as error:
        raise RuntimeError(
            "The graphical selector requires tkinter. Install python3-tk or pass "
            "the policy paths and --config from the command line."
        ) from error

    try:
        root = tk.Tk()
    except tk.TclError as error:
        raise RuntimeError(
            "The graphical selector could not be opened. Make sure a desktop "
            "session is available, or use the command-line arguments."
        ) from error
    root.withdraw()
    root.update()

    try:
        initial_directory = EXPERIMENTS_DIR if EXPERIMENTS_DIR.is_dir() else SCRIPT_DIR
        policies = filedialog.askopenfilenames(
            parent=root,
            title="Select one or more policy files",
            initialdir=str(initial_directory),
            filetypes=[
                ("PyTorch checkpoints", "*.pt *.pth *.ckpt"),
                ("All files", "*"),
            ],
        )
        if not policies:
            raise RuntimeError("No policy file was selected.")

        config = filedialog.askopenfilename(
            parent=root,
            title="Select trajectory.json",
            initialdir=str(initial_directory),
            initialfile=Path(config_path).name,
            filetypes=[
                ("JSON files", "*.json"),
                ("All files", "*"),
            ],
        )
        if not config:
            raise RuntimeError("No trajectory configuration was selected.")
    finally:
        root.destroy()

    return list(policies), Path(config)


def parse_args():
    """Build and parse the evaluator command-line arguments."""
    parser = argparse.ArgumentParser(description="Evaluate manual-automaton DQN policies for LunarLander.")
    parser.add_argument(
        "policies",
        nargs="*",
        help="Checkpoint filenames or explicit checkpoint paths. If omitted, graphical file selectors are opened.",
    )
    parser.add_argument("--config", type=Path, default=SCRIPT_DIR / "trajectory.json", help="Experiment JSON configuration.")
    parser.add_argument("--policy-dir", type=Path, default=SCRIPT_DIR / "policy", help="Directory used to resolve checkpoint filenames.")
    parser.add_argument("--gui", action="store_true", help="Select policies and trajectory.json using graphical dialogs.")
    parser.add_argument("--episodes", type=_positive_int, default=100)
    parser.add_argument("--window", type=_positive_int, default=10)
    parser.add_argument("--seed", type=int, default=None)
    parser.add_argument("--render", action="store_true")
    parser.add_argument("--no-limit", action="store_true", help="Increase the environment episode limit to 5000 steps.")
    parser.add_argument(
        "--trace-grid",
        action="store_true",
        help="Save the sequence of abstract cells visited during evaluation.",
    )
    parser.add_argument(
        "--trace-episodes",
        type=_positive_int,
        default=1,
        help="Number of episodes to trace when --trace-grid is enabled (default: 1).",
    )
    parser.add_argument("--output-dir", type=Path, default=SCRIPT_DIR / "img" / "evaluation")
    parser.add_argument(
        "--network-type",
        choices=["standard", "dueling"],
        default="standard",
        help="Q-network architecture used by the checkpoint.",
    )
    return parser.parse_args()


# ==============================
# Main program
# ==============================

def main():
    """Load the configuration, evaluate the policies, and generate the plots."""
    args = parse_args()
    if args.render and args.trace_grid:
        raise SystemExit(
            "--render and --trace-grid cannot be used together because Gymnasium "
            "requires a single render mode. Run them as separate evaluations."
        )

    # Open native file dialogs when requested or when no policy was supplied.
    if args.gui or not args.policies:
        try:
            args.policies, args.config = _select_files_graphically(args.policy_dir, args.config)
        except RuntimeError as error:
            raise SystemExit(f"Selection cancelled: {error}") from error

    # Load the manual task shared with the trainer.
    with args.config.expanduser().open(encoding="utf-8") as config_file:
        config = json.load(config_file)

    raw_waypoints = config["waypoints_dict"]
    waypoints_dict = {name: tuple(coordinates) for name, coordinates in raw_waypoints.items()}
    grid_w = int(config.get("grid_w", 12))
    grid_h = int(config.get("grid_h", 12))
    goal_reward = float(config.get("goal_reward", 10000.0))
    waypoint_cycle = config.get("waypoint_cycle", list(waypoints_dict))

    # Evaluate policies one at a time to keep rendering and output deterministic.
    results = []
    for policy in args.policies:
        traced_episodes = min(args.trace_episodes, args.episodes) if args.trace_grid else 0
        result = evaluate_policy(
            policy,
            args.policy_dir,
            args.episodes,
            args.render,
            waypoints_dict,
            goal_reward,
            grid_w,
            grid_h,
            args.seed,
            trace_episodes=traced_episodes,
            network_type=args.network_type,
            waypoint_cycle=waypoint_cycle,
            no_limit=args.no_limit,
        )
        results.append(result)

    # Print the summary and create one plot for each evaluated policy.
    args.output_dir.mkdir(parents=True, exist_ok=True)
    for result in results:
        success_rate = result["successes"] / args.episodes
        mean_gym_return = np.mean(result["environment_returns"])
        mean_length = np.mean(result["episode_lengths"])
        mean_cycles = np.mean(result["completed_cycles"])
        reached = ", ".join(f"q={q}: {count}/{args.episodes}" for q, count in result["state_reach_counts"].items())
        print(
            f"[{result['policy']}] success={success_rate:.1%}, "
            f"mean cycles={mean_cycles:.2f}, mean Gym return={mean_gym_return:.2f}, "
            f"mean length={mean_length:.1f} | reached: {reached}"
        )
        print(f"Plot saved to: {plot_policy(result, args.window, args.output_dir)}")
        if args.trace_grid:
            trace_paths = plot_grid_traces(
                result, waypoints_dict, grid_w, grid_h, args.output_dir
            )
            for episode_index, (cells, trace_path) in enumerate(
                zip(result["grid_traces"], trace_paths), start=1
            ):
                waypoint_status = format_waypoint_trace(cells, waypoints_dict)
                print(
                    f"Grid trace episode {episode_index}: {waypoint_status} | "
                    f"saved to: {trace_path}"
                )

    # Add a combined comparison when more than one policy was requested.
    if len(results) > 1:
        print(f"Comparison saved to: {plot_comparison(results, args.window, args.output_dir)}")


if __name__ == "__main__":
    main()


## 10. Configure the experiment

The Gym reward remains ignored by the trainer. The best policy is selected
using the mean learning reward over the logging window.


In [ ]:
EPISODES = 1000
EPSILON_DECAY = 0.9996
SHAPING_SCALE = 1.0
LOG_INTERVAL = 100
PLOT_WINDOW = 500
DISABLE_SHAPING = False
USE_POLYAK = True
POLYAK_TAU = 0.005
TARGET_UPDATE_FREQ = 1000  # Used only when USE_POLYAK is False
NETWORK_TYPE = "standard"  # Use "dueling" for Dueling DDQN

print(f"Episodes: {EPISODES}")
print(f"Epsilon decay: {EPSILON_DECAY}")
print(f"Shaping scale: {SHAPING_SCALE}")
print(f"Log interval: {LOG_INTERVAL}")
print(f"Polyak update: {USE_POLYAK}")
print(f"Network type: {NETWORK_TYPE}")
if USE_POLYAK:
    print(f"Polyak tau: {POLYAK_TAU}")
else:
    print(f"Hard target update frequency: {TARGET_UPDATE_FREQ}")


## 11. Run training


In [ ]:
import os
import subprocess
import sys

command = [
    sys.executable,
    "-u",
    "trainer.py",
    "--episodes",
    str(EPISODES),
    "--config",
    "trajectory.json",
    "--eps-decay",
    str(EPSILON_DECAY),
    "--shaping-scale",
    str(SHAPING_SCALE),
    "--log-interval",
    str(LOG_INTERVAL),
    "--plot-window",
    str(PLOT_WINDOW),
    "--polyak-tau",
    str(POLYAK_TAU),
    "--target-update-freq",
    str(TARGET_UPDATE_FREQ),
    "--network-type",
    NETWORK_TYPE,
]
if DISABLE_SHAPING:
    command.append("--no-shaping")
if not USE_POLYAK:
    command.append("--no-polyak")

environment = os.environ.copy()
environment["MPLBACKEND"] = "Agg"
environment["PYTHONUNBUFFERED"] = "1"
print("Running:", " ".join(command))
process = subprocess.Popen(
    command,
    cwd=WORK_DIR,
    env=environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)


## 12. Inspect metrics, logs, policies, and generated plots


In [ ]:
import numpy as np
from IPython.display import Image, display

data_path = WORK_DIR / "results" / "single_epsilon_data.npz"
metrics = np.load(data_path, allow_pickle=False)

print("Saved metrics:")
for key in metrics.files:
    value = metrics[key]
    print(f"- {key}: shape={value.shape}, dtype={value.dtype}")

print(f"\nBest policy episode: {int(metrics['best_policy_episode'])}")
print(
    "Best mean learning reward: "
    f"{float(metrics['best_mean_learning_reward']):.3f}"
)
print(f"Overall success rate: {metrics['successes'].mean():.2%}")
print(f"Mean completed cycles: {metrics['completed_cycles'].mean():.3f}")

log_path = WORK_DIR / "logs" / "single_epsilon_training.log"
if log_path.exists():
    print("\nLast log lines:\n")
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-30:]))

print("\nSaved policies:")
for policy_path in sorted((WORK_DIR / "policy").glob("*.pth")):
    print(f"- {policy_path.name}")

plot_paths = [
    WORK_DIR / "img" / "alternating_goals_automaton.png",
    WORK_DIR / "img" / "buffer_fractions_single_epsilon.png",
    WORK_DIR / "img" / "reward_breakdown_single_epsilon.png",
    *sorted((WORK_DIR / "img" / "heatmaps").glob("*.png")),
]
for plot_path in plot_paths:
    if plot_path.is_file():
        print(f"\n{plot_path.relative_to(WORK_DIR)}")
        display(Image(filename=str(plot_path)))


## 13. Package outputs for download


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

archive_path = Path("/kaggle/working/lunar_lander_manual_outputs.zip")
output_directories = ("results", "img", "logs", "policy")

with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for directory_name in output_directories:
        output_directory = WORK_DIR / directory_name
        if output_directory.exists():
            for output_path in sorted(output_directory.rglob("*")):
                if output_path.is_file():
                    archive.write(
                        output_path,
                        output_path.relative_to(WORK_DIR),
                    )
    configuration_path = WORK_DIR / "trajectory.json"
    if configuration_path.is_file():
        archive.write(configuration_path, configuration_path.name)

print(f"Output archive ready: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / (1024 ** 2):.2f} MB")
